# 02 — 推薦モデルの挙動を理解する

## 1. Overview

- ユーザーの履歴と推薦曲を見比べ、選ばれた理由・人気度の偏りを確認する。
- モデル学習・推薦・分析用データの作成は `src/` を呼び出す。

| Notebook | 目的 |
| --- | --- |
| 01 | 保存済みデータの検証・EDA |
| 02 | 推薦の実行・モデルの挙動の理解 |
| 03 | 共通指標でモデルの性能を比較 |

| model | 仕組み | 状態 |
| --- | --- | --- |
| `popularity` | 未聴候補を学習全体の再生回数で順位付け。同数は曲ID昇順 | 実装済み |
| `llm` | LLMによる推薦 | 将来追加 |
| `agent` | ツールを使うAgentによる推薦 | 将来追加 |

- 既定は6曲の合成データ。曲名・アーティスト・タグ・音源は含まれない。
- `MUSIC_RECSYS_DATASET_PATH` で保存済みの評価入力JSONを指定できる。

## 2. Setup

| 設定 | 用途 |
| --- | --- |
| `MODEL` | モデル選択。未実装モデルはエラーで通知 |
| `MUSIC_RECSYS_DATASET_PATH` | 学習履歴・要求を含む保存済みJSON。未指定はtoy |
| `MUSIC_RECSYS_REQUEST_ID` | 要求ID。未指定は実データの先頭要求／toyのr1 |
| `MUSIC_RECSYS_TOP_K` | 推薦曲数。既定は実データ10／toy 2 |
| `AUDIO_FILES` | このデータセットの楽曲IDとローカル音源の対応 |

- 設定を変更したら、この章から再実行する。各Notebookは独立して実行可能。
- LLM・Agentの実装時は共通のモデル選択に追加し、同じ推薦呼出しを使う。
- ユーザーID・履歴はローカルで確認する。共有時は出力を消去する。

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from music_recsys.data import load_dataset
from music_recsys.recommendation_inspection import (
    audio_player,
    inspect_recommendation,
    request_options,
)

ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
MODEL = "popularity"
saved_dataset_path = os.environ.get("MUSIC_RECSYS_DATASET_PATH")
DATASET_PATH = (
    Path(saved_dataset_path) if saved_dataset_path else ROOT / "examples/toy/dataset.json"
)
REQUEST_ID = os.environ.get("MUSIC_RECSYS_REQUEST_ID")
TOP_K = int(os.environ.get("MUSIC_RECSYS_TOP_K", "10" if saved_dataset_path else "2"))
# 利用可能な音源を指定: {"楽曲ID": Path("/absolute/path/to/audio.wav")}
AUDIO_FILES = {}

dataset = load_dataset(DATASET_PATH, top_k=TOP_K)
if REQUEST_ID is None:
    REQUEST_ID = dataset.requests[0].request_id if saved_dataset_path else "r1"
display(pd.DataFrame(request_options(dataset)))
display(
    pd.DataFrame(
        [
            {
                "dataset": dataset.dataset_id,
                "sha256": dataset.sha256,
                "model": MODEL,
                "top_k": TOP_K,
                "train_events": len(dataset.train_interactions),
            }
        ]
    )
)

## 3. Recommendation

- 選択した要求を `inspect_recommendation` に渡す。
- モデルへの入力は学習履歴・推薦要求。正解データは渡さない。
- 評価指標の計算・モデル間比較は03で行う。

In [ ]:
result = inspect_recommendation(dataset, model=MODEL, request_id=REQUEST_ID)
display(
    pd.DataFrame(
        [
            {
                "model": result.model,
                "request_id": result.request.request_id,
                "user_id": result.request.user_id,
                "history_count": len(result.request.history),
                "candidate_count": len(result.request.candidate_ids),
                "returned_count": len(result.response.item_ids),
                "status": result.response.status,
            }
        ]
    )
)

## 4. Result Inspection

### ユーザーの履歴と推薦曲

| 列 | 意味 |
| --- | --- |
| `user_train_events` | このユーザーの学習期間中の再生回数 |
| `train_events` | 学習全体の再生回数 |
| `score` | Popularityの順位付けに使う再生回数。好みの確率ではない |
| `reason` | 現モデルの選択ルールに基づく根拠 |
| `in_history` | 推薦時に渡した履歴に含まれるか |

- 履歴は推薦時点の既聴曲。この表は曲単位の集計で、表示順は時系列を表さない。
- 履歴がないユーザーは空表。未聴候補がK未満なら推薦数もK未満。

In [ ]:
history_df = pd.DataFrame(result.history, columns=["item_id", "user_train_events", "train_events"])
recommendations_df = pd.DataFrame(
    result.recommendations,
    columns=["rank", "item_id", "train_events", "in_history", "score", "reason"],
)
display(history_df.rename_axis("history_row"))
with pd.option_context("display.max_colwidth", None):
    display(recommendations_df)

display(
    pd.DataFrame(
        [
            {
                "history_overlap": int(recommendations_df["in_history"].sum()),
                "requested": TOP_K,
                "returned": len(recommendations_df),
                "short_output": len(recommendations_df) < TOP_K,
            }
        ]
    )
)

### 人気度・候補との比較

- 履歴の図は学習全体の再生回数が多い上位20曲に限定。推薦曲と人気度を比較する。
- 候補表で、推薦された曲と選ばれなかった曲の差を確認する。
- Popularityは全体の人気を利用し、ユーザーの履歴は既聴曲の除外に使う。嗜好の類似度は計算しない。

In [ ]:
candidates_df = pd.DataFrame(result.candidates)
display(candidates_df)

history_plot = history_df.sort_values(["train_events", "item_id"], ascending=[False, True]).head(20)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, frame, title, color in [
    (axes[0], history_plot, "History (top 20 by training events)", "#627d98"),
    (axes[1], recommendations_df, "Recommendations", "#2a9d8f"),
]:
    if frame.empty:
        ax.text(0.5, 0.5, "No tracks", ha="center", transform=ax.transAxes)
    else:
        ax.bar(frame["item_id"], frame["train_events"], color=color)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.set_title(title)
    ax.set_xlabel("Track ID")
axes[0].set_ylabel("Training events (all users)")
fig.tight_layout()
plt.show()

### 試聴

- `AUDIO_FILES` に音源を登録し、`AUDITION_ITEM_ID` で履歴曲・推薦曲を選ぶ。
- 再生ボタンで試聴。未登録の曲は「音源未登録」と表示する。
- 音源の自動取得は行わない。合成IDへの音源割当はプレイヤーの動作確認用。

In [ ]:
AUDITION_ITEM_ID = result.response.item_ids[0] if result.response.item_ids else None
if AUDITION_ITEM_ID is None:
    print("推薦曲なし。試聴する履歴曲のIDを指定できます。")
else:
    player = audio_player(AUDITION_ITEM_ID, AUDIO_FILES)
    display(
        pd.DataFrame(
            [
                {
                    "item_id": AUDITION_ITEM_ID,
                    "audio": "再生可能" if player is not None else "音源未登録",
                }
            ]
        )
    )
    if player is not None:
        display(player)

### 将来の確認項目

| 対象 | 確認したい内容 | 現在 |
| --- | --- | --- |
| 楽曲情報 | 曲名・アーティスト・タグと履歴の比較 | 推薦データに未接続 |
| LLM | 生成された推薦理由 | 未実装 |
| Agent | ツール呼出し・結果・推薦理由 | 未実装 |

- LangChain・LangGraphの採用はLLM・Agent実装時に判断する。

## 5. Summary

- モデル・要求を切り替え、推薦曲・根拠・履歴との重複を確認する。
- 次の表に観察を記入し、性能比較は03で行う。

| 観点 | 観察メモ |
| --- | --- |
| 履歴と推薦曲の違い | — |
| 人気度・選択ルールへの気づき | — |
| 試聴した印象 | — |
| 次に試すユーザー・モデル | — |